# Data Access Layer (API) Testing

This notebook verifies the four typed access functions required by the intern brief. It tests for correctness, edge-case handling, and adversarial timestamp support.

In [1]:
import pandas as pd
import sys
import os

# Ensure we can import from src/
sys.path.append(os.path.abspath('.'))

from src.api import DataAPI

api = DataAPI()

## 1. get_price(timestamp)
Fetches the exact 1-minute OHLCV for Spot and both Near/Mid Futures.

In [2]:
ts = pd.to_datetime('2025-08-22 10:15:00')
print("Requesting valid timestamp: 10:15:00")
display(api.get_price(ts))

ts_pre = pd.to_datetime('2025-08-22 09:05:00')
print("\nRequesting pre-open (adversarial) timestamp: 09:05:00")
display(api.get_price(ts_pre)) # Should return empty safely

Requesting valid timestamp: 10:15:00


,timestamp,spot_open,spot_high,spot_low,spot_close,spot_volume,fut_near_open,fut_near_close,fut_mid_open,fut_mid_close
0,2025-08-22 10:15:00,24063.64,24064.97,24052.28,24053.75,355208,24080.49,24080.19,24198.32,24196.97



Requesting pre-open (adversarial) timestamp: 09:05:00


,timestamp,spot_open,spot_high,spot_low,spot_close,spot_volume,fut_near_open,fut_near_close,fut_mid_open,fut_mid_close


## 2. get_signals(timestamp)
Fetches the Options Chain snapshot. Implements **AS-OF Join** logic (looks back up to 5 mins if requested between snapshots).

In [3]:
print("Requesting exact snapshot: 10:15:00")
df_exact = api.get_signals(pd.to_datetime('2025-08-22 10:15:00'))
print(f"Returned {len(df_exact)} rows.")

print("\nRequesting between snapshots (AS-OF): 10:17:00")
df_asof = api.get_signals(pd.to_datetime('2025-08-22 10:17:00'))
print(f"Returned {len(df_asof)} rows. Validated Snapshot TS: {df_asof['timestamp'].iloc[0] if not df_asof.empty else 'None'}")
display(df_asof.head())

Requesting exact snapshot: 10:15:00
Returned 116 rows.

Requesting between snapshots (AS-OF): 10:17:00
Returned 116 rows. Validated Snapshot TS: 2025-08-22 10:15:00


,timestamp,strike,side,open,high,low,close,volume,oi,iv,expiry,anomaly_code,snapshot_id,expiry_str,trade_date
0,2025-08-22 10:15:00,23500,CE,927.96,948.70,938.31,941.50,181,30143,0.1446,2025-09-25,0,93,2025-09-25,2025-08-22
1,2025-08-22 10:15:00,23500,PE,416.95,421.57,416.69,416.88,133,31924,0.1428,2025-09-25,0,93,2025-09-25,2025-08-22
2,2025-08-22 10:15:00,23550,CE,938.59,933.35,926.92,932.20,39,29985,0.1456,2025-09-25,0,93,2025-09-25,2025-08-22
3,2025-08-22 10:15:00,23550,PE,420.54,429.46,424.86,426.23,558,30185,0.1448,2025-09-25,0,93,2025-09-25,2025-08-22
4,2025-08-22 10:15:00,23600,CE,886.78,897.21,886.83,888.35,16,29219,0.1464,2025-09-25,0,93,2025-09-25,2025-08-22


## 3. get_features(timestamp)
Returns a historical feature matrix (30-min lookback returns).

In [4]:
ts = pd.to_datetime('2025-08-22 11:00:00')
print("Generating 30-min lookback features for 11:00 AM")
features = api.get_features(ts)
display(features.head())
print(f"Feature matrix rows: {len(features)}")

Generating 30-min lookback features for 11:00 AM


,timestamp,close,volume,anomaly_code,returns
0,2025-08-22 11:00:00,24001.63,311815,0,NaN
1,2025-08-22 10:59:00,23995.88,314989,0,-0.000240
2,2025-08-22 10:58:00,23998.29,62355,0,0.000100
3,2025-08-22 10:57:00,24009.93,317567,0,0.000485
4,2025-08-22 10:56:00,24014.95,123970,0,0.000209


Feature matrix rows: 30


## 4. get_features_batch(timestamps)
High-throughput retrieval optimized using in-memory DuckDB joins.

In [5]:
batch = [
    pd.to_datetime('2025-08-22 10:15:00'),
    pd.to_datetime('2025-08-22 10:16:00'),
    pd.to_datetime('2025-08-22 10:17:00'),
    pd.to_datetime('2026-01-01 00:00:00') # Adversarial Future TS
]

print("Executing Batch Retrieval...")
df_batch = api.get_features_batch(batch)
display(df_batch)
print("Note how the 2026 timestamp returns NaNs gracefully, preserving input alignment.")

Executing Batch Retrieval...


,timestamp,spot_close,spot_volume,near_fut_close,vix_close
0,2025-08-22 10:15:00,24053.75,355208,24080.19,13.99
1,2025-08-22 10:16:00,24050.99,298323,24077.14,14.02
2,2025-08-22 10:17:00,24034.94,177325,24061.82,14.05
3,2026-01-01 00:00:00,NaN,<NA>,NaN,NaN


Note how the 2026 timestamp returns NaNs gracefully, preserving input alignment.
